# Defter 21 — Climate-change attribution, limiting factors, growth-curve shift and uncertainty with the final model
**Theory-guided AI reveals how climate extremes reshape global tree growth** · Prof. Dr. İlker Ercanlı

**Ne yapar (TR):** Yeni eğitim yok. Defter 20'nin nihai modelini (`teori_vpd_opt_re`, 2×128; 4 kat × 5 tohum = 20 üye)
yükler ve şu soruları yanıtlar:

| Bölüm | Soru | Çıktı |
|---|---|---|
| Q0 | İklim ne kadar değişti? (1990–2019 vs 1901–1960; gözlenen − karşı-olgusal) | `Q0_iklim_degisimi_alan.csv`, `Q0_iklim_degisimi_grup.csv` |
| Q1 | Görülmemiş sahalarda belirsizlik düzeltmesi (yeni ağaçların ağaç etkisi varyansı σ'ya eklenir) | `Q1_saha_kalibrasyon.csv` |
| Q2 | On yıllık sistematik sapma nereden geliyor? (bölge, cins, kuraklık, ağaç yaşı, boyut, örnekleme yılı) | `Q2_sapma_grup.csv`, `Q2_sapma_yas_onyil.csv` |
| Q3 | İklim değişikliği artımı nerede, hangi on yılda, ne kadar etkiledi? Sıcaklık / nem / VPD payları | `Q3_atif_onyil.csv`, `Q3_atif_alan.csv`, `Q3_atif_grup.csv`, `Q3_atif_kanal.csv` |
| Q4 | Sınırlayıcı faktör ve rejim kayması (sıcaklıktan neme geçiş) | `Q4_sinirlayici_alan.csv`, `Q4_sinirlayici_grup.csv` |
| Q5 | Büyüme eğrisi kayması (referans ağaç, 100 yıl; olgusal vs karşı-olgusal; 1931–60 vs 1990–2019 iklimi) | `Q5_buyume_egrisi_alan.csv`, `Q5_buyume_egrisi_grup.csv`, `Q5_egri_ortalama.csv` |

**Önce:** Defter 20 bitmiş olmalı (`ITRDB_AI/SONUC_20/`), `ITRDB_AI/TENSOR/veri_v2.npz`, ve Defter 11'in kullandığı
`ITRDB_AI/IKLIM_ISIMIP/isimip_obsclim_aylik.csv.gz` ile `isimip_counterclim_aylik.csv.gz`. İnternetten indirme yok.
**Çalışma zamanı:** L4 (yoksa T4) GPU, yüksek RAM → "Tümünü çalıştır". Ara sonuçlar kaydedilir; kesilirse yeniden çalıştırınca biten adımlar atlanır.
**Çıktılar:** `ITRDB_AI/SONUC_21/`. **Bitiş satırı:** `==== DEFTER 21 TAMAM ====`

Not: Etkinin büyüklüğünün bağımsız ölçüsü halkalardan modelsiz hesaptır (makale, denklem 4–5). Buradaki model atfı
"nerede, neden, ne kadar emin" sorularına cevap verir; Q2'deki sapma atfın büyüklüğünü etkileyebilir.

In [ ]:
# --- Setup ---
import os, json, time, math
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ANA = '/content/drive/MyDrive/ITRDB_AI'
except ImportError:                                   # local test outside Colab
    ANA = os.environ.get('ITRDB_AI', '.')
KAYNAK = os.path.join(ANA, 'SONUC_20')                # trained models of Defter 20
CIKTI = os.path.join(ANA, 'SONUC_21'); os.makedirs(CIKTI, exist_ok=True)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEV, torch.cuda.get_device_name(0) if DEV == 'cuda' else '(no GPU: Runtime -> change runtime type -> GPU)')

AYAR = dict(
    J=8, emb=8, takson_dusur=0.10, desc_gurultu=0.2, r0_bas_cm=2.0, sifir_mm=0.005,
    blok_yil=5, n_kat=4, bolme_tohum=2026, ic_blok_oran=0.10, agac_onsel=0.3, hizli_deneme=False,
    model='teori_vpd_opt_re',                        # final model (Defter 20)
    yil_son=2019,                                    # ISIMIP3a obsclim/counterclim end year
    kanal_uyeler=None,                               # None: all 20 members for the channel runs; or e.g. 4
    egri_yil=100, egri_cap_cm=[15, 30],              # growth-curve simulation: years, diameter thresholds
)
AYAR.update(json.loads(os.environ.get('D21_AYAR', '{}')))   # override only for local testing
if AYAR['hizli_deneme']: AYAR['max_epoch'] = min(AYAR['max_epoch'], 2)
t = lambda x, dt=None: torch.as_tensor(np.asarray(x), dtype=dt).to(DEV)

In [ ]:
# --- Data (identical preparation to Defter 2) ---
d = np.load(os.path.join(ANA, 'TENSOR', 'veri_v2.npz'))
soz = json.load(open(os.path.join(ANA, 'TENSOR', 'veri_v2_sozluk.json')))
YIL0 = soz['yil0']; S = len(soz['site_code']); NG = len(soz['genus']); NS = len(soz['species'])
NSER = len(d['series_site']); NTREE = int(d['series_tree'].max()) + 1

clim = d['clim']; soil = d['soil']; gE = d['gE']; NY = clim.shape[1]
YIL1 = YIL0 + NY - 1
# rings need the previous year's climate, so 1901 rings are dropped (as in Defter 2)
k = (d['ring_year'] > YIL0) & (d['ring_year'] <= YIL1)
R_ser = t(d['ring_series'][k], torch.long); R_yil = t(d['ring_year'][k].astype(np.int64), torch.long)
R_w = t(d['ring_rw'][k], torch.float32); R_rp = t(d['ring_rprev'][k], torch.float32) / 10   # cm
ser_site = t(d['series_site'], torch.long); ser_tree = t(d['series_tree'], torch.long)
R_site = ser_site[R_ser]; R_tree = ser_tree[R_ser]
site_genus = t(d['site_genus'], torch.long); site_species = t(d['site_species'], torch.long)
NR = len(R_w)

# monthly climate per site: T, log(1+P), soil moisture M, VPD, growing-season weight gE (+ CWD, SPEI-like, below)
ozel = np.stack([clim[..., 0], np.log1p(np.maximum(clim[..., 1], 0)), soil, clim[..., 3],
                 np.broadcast_to(gE[:, None, :], soil.shape)], -1).reshape(S, NY * 12, 5).astype(np.float32)
# --- additional water-stress variables (computed from the same CRU TS and bucket-model data; no download) ---
# climatic water deficit: CWD = PET - AET, with AET = PET x M / Mmax (supply-limited evapotranspiration of the
# VS-Lite bucket, Mmax = 0.76); in units of 100 mm per month
MMAX = 0.76
pet = np.maximum(np.nan_to_num(clim[..., 2]), 0)
cwd = (pet * (1 - np.clip(soil, 0, MMAX) / MMAX) / 100).reshape(S, NY * 12)
# SPEI-like index: 12-month sum of P - PET, standardized per site and calendar month against 1901-1960
wb = (np.nan_to_num(clim[..., 1]) - pet).reshape(S, NY * 12)
cs = np.cumsum(wb, axis=1); wb12 = np.full_like(cs, np.nan); wb12[:, 11:] = cs[:, 11:] - np.concatenate([np.zeros((S, 1)), cs[:, :-12]], 1)
wb12 = wb12.reshape(S, NY, 12); ref = wb12[:, :60]
z12 = (wb12 - np.nanmean(ref, 1, keepdims=True)) / (np.nanstd(ref, 1, keepdims=True) + 1e-6)
z12 = np.nan_to_num(np.clip(z12, -4, 4)).reshape(S, NY * 12)
ozel = np.concatenate([ozel, cwd[..., None].astype(np.float32), z12[..., None].astype(np.float32)], -1)
print(f'Water deficit: median {np.median(cwd)*100:.0f} mm/month; SPEI-like index range {z12.min():.1f} to {z12.max():.1f}')
del pet, cwd, wb, cs, wb12, ref, z12
n_nan = int(np.isnan(ozel).sum())
if n_nan: print(f'WARNING: {n_nan} missing climate values set to the site-month mean')
if n_nan:
    nk = ozel.shape[-1]
    ort = np.nanmean(ozel.reshape(S, NY, 12, nk), axis=1, keepdims=True)
    o4 = ozel.reshape(S, NY, 12, nk); o4[:] = np.where(np.isnan(o4), ort, o4); ozel = o4.reshape(S, NY * 12, nk)
OZ = t(ozel); del ozel

desc = d['desc']; DESC = t((desc - desc.mean(0)) / desc.std(0), torch.float32)
env = d['env']; ENV = t((env - env.mean(0)) / (env.std(0) + 1e-6), torch.float32)
site_biyom = t(d['site_biyom'], torch.long); site_eko = t(d['site_eko'], torch.long); site_kg = t(d['site_kg'], torch.long)
NB, NE, NK = soz['n_biyom'], soz['n_eko'], soz['n_kg']
IC = t(d['site_ic_dogrulama'], torch.bool)            # 10% of sites reserved for early stopping (fixed in Defter 1)
print(f'{NR:,} rings, {NSER:,} series, {NTREE:,} trees, {S} sites, years {YIL0+1}-{YIL1}')

def pencere(idx):
    """24-month climate window (previous January - current December) for rings idx: [B, 24, 5]"""
    s = R_site[idx]; bas = (R_yil[idx] - 1 - YIL0) * 12
    ay = bas[:, None] + torch.arange(24, device=DEV)[None, :]
    return OZ[s[:, None], ay], s

def girdiler(idx):
    X, s = pencere(idx)
    return dict(X=X, rp=R_rp[idx], tree=R_tree[idx], g=site_genus[s], sp=site_species[s], desc=DESC[s],
                env=ENV[s], b=site_biyom[s], e=site_eko[s], k=site_kg[s])

In [ ]:
# --- Year-block split: 5-year blocks; within each run of 4 consecutive blocks the folds are permuted,
#     so every fold covers the whole period (early, mid-century and the recent warm decades) ---
yillar = np.arange(YIL0 + 1, YIL1 + 1)
blok = (yillar - yillar[0]) // AYAR['blok_yil']; nb = int(blok.max()) + 1
rng = np.random.default_rng(AYAR['bolme_tohum']); kat_blok = np.empty(nb, int)
for b0 in range(0, nb, AYAR['n_kat']):
    ix = np.arange(b0, min(b0 + AYAR['n_kat'], nb)); kat_blok[ix] = rng.permutation(AYAR['n_kat'])[:len(ix)]
yk = np.full(YIL1 + 2, -1, np.int64); yk[yillar] = kat_blok[blok]
YK = t(yk, torch.long)
tablo = pd.DataFrame(dict(year=yillar, block=blok, fold=kat_blok[blok]))
SITE_KAT = t(d['site_blok_kat'], torch.long)          # 10-degree spatial blocks in 5 groups (fixed in Defter 1)
R_kat = YK[R_yil]
for kk in range(AYAR['n_kat']):
    yy = tablo[tablo.fold == kk].year.to_numpy()
    print(f'fold {kk}: {len(yy)} years ({len(yy)/len(yillar):.0%}), {(R_kat == kk).float().mean().item():.1%} of rings; '
          f'blocks starting ' + ', '.join(str(y) for y in yy[::AYAR["blok_yil"]]))

bk = np.full(YIL1 + 2, -1, np.int64); bk[yillar] = blok
R_blok = t(bk, torch.long)[R_yil]

def bolunme(kat):
    """Test: this fold's years (~25%) at all sites. Early stopping: ~10% of the remaining 5-year blocks, drawn at
    random, at all sites (unseen years, as in the test). Train: all other years at all sites.
    No test year is used for training or for early stopping."""
    egit_blok = np.unique(blok[kat_blok[blok] != kat])
    n_ic = max(2, int(round(AYAR['ic_blok_oran'] * len(egit_blok))))
    ic_blok = np.random.default_rng(AYAR['bolme_tohum'] + 100 + kat).choice(egit_blok, n_ic, replace=False)
    test = R_kat == kat; ic = torch.isin(R_blok, t(ic_blok, torch.long))
    egit = (~test & ~ic).nonzero().squeeze(1); dog = (~test & ic).nonzero().squeeze(1); tst = test.nonzero().squeeze(1)
    return _alt(egit, dog, tst)

def bolunme_saha(kat):
    """Unseen-site test: test = all years at the sites of spatial group `kat` (10-degree blocks); early stopping =
    the fixed 10% validation sites outside the test group (also unseen sites); train = all other sites, all years.
    Trees at test sites were never seen, so their tree effect stays at its prior mean (zero)."""
    ts = SITE_KAT[R_site] == kat; ic = IC[R_site] & ~ts
    return _alt((~ts & ~ic).nonzero().squeeze(1), ic.nonzero().squeeze(1), ts.nonzero().squeeze(1))

def _alt(egit, dog, tst):
    if AYAR['hizli_deneme']:
        g = torch.Generator(device='cpu').manual_seed(0)
        sec = lambda x, n: x[torch.randperm(len(x), generator=g)[:n].to(DEV)]
        egit, dog = sec(egit, 200_000), sec(dog, 20_000)
    return egit, dog, tst

In [ ]:
# --- Model: theory-embedded network (equations 1-3 of the manuscript) with options ---
class MLP(nn.Module):
    """Site-parameter network with `katman` hidden layers of `gizli` units (residual blocks beyond the first)."""
    def __init__(s, n_in, gizli, katman, n_out):
        super().__init__()
        s.gir = nn.Linear(n_in, gizli)
        s.bloklar = nn.ModuleList([nn.Sequential(nn.LayerNorm(gizli), nn.Linear(gizli, gizli), nn.GELU()) for _ in range(katman - 1)])
        s.cik = nn.Linear(gizli, n_out)
        with torch.no_grad(): s.cik.weight.mul_(0.1); s.cik.bias.zero_()
    def forward(s, x):
        h = F.gelu(s.gir(x))
        for b in s.bloklar: h = h + b(h)
        return s.cik(h)

class SiteAgi(nn.Module):
    """Genus/species embeddings + site descriptors + site environment (soil, topography, biome, ecoregion, Koppen)."""
    def __init__(s, n_out, gizli, katman):
        super().__init__()
        e = AYAR['emb']
        s.ge = nn.Embedding(NG + 1, e); s.se = nn.Embedding(NS + 1, e)              # last index = unknown taxon
        s.b = nn.Embedding(NB + 1, 4); s.e = nn.Embedding(NE + 1, 4); s.k = nn.Embedding(NK + 1, 4)
        s.mlp = MLP(2 * e + DESC.shape[1] + ENV.shape[1] + 12, gizli, katman, n_out)
    def forward(s, G, gizle=None):
        g, sp, b, e, k = G['g'], G['sp'], G['b'], G['e'], G['k']
        if gizle is not None:                                                   # taxon dropout: learn the 'unknown' embedding
            g = torch.where(gizle, torch.full_like(g, NG), g); sp = torch.where(gizle, torch.full_like(sp, NS), sp)
        if s.training:                                                          # 10% masking of the categorical environment
            m = torch.rand(len(b), device=DEV) < 0.10
            b = torch.where(m, torch.full_like(b, NB), b); e = torch.where(m, torch.full_like(e, NE), e); k = torch.where(m, torch.full_like(k, NK), k)
        dn = G['desc'] + (AYAR['desc_gurultu'] * torch.randn_like(G['desc']) if s.training else 0)
        en = G['env'] + (AYAR['desc_gurultu'] * torch.randn_like(G['env']) if s.training else 0)
        return s.mlp(torch.cat([s.ge(g), s.se(sp), dn, en, s.b(b), s.e(e), s.k(k)], 1))

def monoton(x, a, c, s):
    """Equation (3): non-negative mixture of logistic functions, rising from 0 to 1. x:[B,24], a,c,s:[B,J]"""
    w = F.softplus(a) + 1e-4
    z = torch.sigmoid((x[..., None] - c[:, None, :]) / s[:, None, :])
    return (z * w[:, None, :]).sum(-1) / w.sum(-1, keepdim=True)

class TeoriModeli19(nn.Module):
    def __init__(s, vpd=True, opt=False, iklim=True, agac=False, cwd=False, spei=False, gizli=128, katman=2):
        super().__init__()
        J = AYAR['J']; s.J, s.vpd, s.opt, s.iklim, s.agac, s.cwd, s.spei = J, vpd, opt, iklim, agac, cwd, spei
        n = 3 + 6 * J + 24 + (3 * J if vpd else 0) + (2 if opt else 0) + (3 * J if cwd else 0) + (3 * J if spei else 0) + 3 + 1
        s.ag = SiteAgi(n, gizli, katman)
        s.Tbaz = torch.linspace(-5, 25, J, device=DEV); s.Mbaz = torch.linspace(0.02, 0.6, J, device=DEV)
        s.Dbaz = torch.linspace(0.2, 3.0, J, device=DEV)
        s.Wbaz = torch.linspace(0.05, 1.5, J, device=DEV); s.Zbaz = torch.linspace(-2.5, 1.5, J, device=DEV)
        s.u = nn.Parameter(torch.zeros(NTREE, device=DEV))                     # tree effect (used only when agac=True)
        s.m_logit = math.log((2 / 3) / (1 - 2 / 3))

    def parametreler(s, G, gizle=None):
        o = s.ag(G, gizle); J = s.J; p = {}
        p['eta'] = torch.exp(o[:, 0] + math.log(0.7))
        p['Bmax'] = torch.exp(o[:, 1] + math.log(1e4))                         # cm^2
        p['m'] = torch.sigmoid(o[:, 2] + s.m_logit)                            # starts at 2/3 (metabolic scaling)
        i = 3
        p['aT'], p['cT'], p['sT'] = o[:, i:i+J], s.Tbaz + 5 * torch.tanh(o[:, i+J:i+2*J]), 0.5 + 2 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['aM'], p['cM'], p['sM'] = o[:, i:i+J], s.Mbaz + 0.1 * torch.tanh(o[:, i+J:i+2*J]), 0.005 + 0.03 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['omega'] = torch.softmax(o[:, i:i+24], 1); i += 24
        if s.vpd:
            p['aD'], p['cD'], p['sD'] = o[:, i:i+J], s.Dbaz + 0.3 * torch.tanh(o[:, i+J:i+2*J]), 0.05 + 0.3 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        if s.cwd:   # climatic water deficit (100 mm/month); response decreases with deficit
            p['aW'], p['cW'], p['sW'] = o[:, i:i+J], s.Wbaz + 0.2 * torch.tanh(o[:, i+J:i+2*J]), 0.05 + 0.3 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        if s.spei:  # 12-month standardized water balance; response increases with wetter conditions
            p['aZ'], p['cZ'], p['sZ'] = o[:, i:i+J], s.Zbaz + 0.5 * torch.tanh(o[:, i+J:i+2*J]), 0.2 + 0.5 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        if s.opt:   # temperature optimum: growth declines above a site-specific upper threshold (starts near 25 °C)
            p['cH'], p['sH'] = 25 + 7 * torch.tanh(o[:, i]), 0.5 + 2 * F.softplus(o[:, i+1]); i += 2
        p['sig0'], p['sigC'], p['sigR'] = o[:, i] - 0.3, 0.1 * o[:, i+1], 0.1 * o[:, i+2]; i += 3   # heteroscedastic SD
        p['r0'] = F.softplus(o[:, i] + math.log(math.expm1(AYAR['r0_bas_cm']))); i += 1
        return p

    def forward(s, idx, gizle=None, G=None):
        G = girdiler(idx) if G is None else G
        X = G['X']; p = s.parametreler(G, gizle)
        r = p['r0'] + G['rp']; B = math.pi * r ** 2
        x = (B / p['Bmax']).clamp_min(1e-8) ** (1 - p['m'])
        Gp = p['eta'] * B.clamp_min(1e-6) ** p['m'] * F.softplus(20 * (1 - x)) / 20       # equation (1), smoothed
        T, M, D, gEk = X[..., 0], X[..., 2], X[..., 3], X[..., 4]
        if s.iklim:
            gT = monoton(T, p['aT'], p['cT'], p['sT'])
            if s.opt: gT = gT * (1 - torch.sigmoid((T - p['cH'][:, None]) / p['sH'][:, None]))
            gM = monoton(M, p['aM'], p['cM'], p['sM'])
            sinir = torch.minimum(gT, gM)
            gD = 1 - monoton(D, p['aD'], p['cD'], p['sD']) if s.vpd else torch.ones_like(gT)
            sinir = torch.minimum(sinir, gD)                                   # Liebig: min(gT, gM, gD), equation (2)
            if s.cwd: sinir = torch.minimum(sinir, 1 - monoton(X[..., 5], p['aW'], p['cW'], p['sW']))
            if s.spei: sinir = torch.minimum(sinir, monoton(X[..., 6], p['aZ'], p['cZ'], p['sZ']))
        else:
            gT = gM = gD = sinir = torch.ones_like(T)                          # climate-free ablation
        C = (p['omega'] * gEk * sinir).sum(1).clamp_min(1e-6)
        u = s.u[G['tree']] if s.agac else torch.zeros_like(C)
        BAI = Gp * C * torch.exp(u)
        w_hat = (torch.sqrt((B + BAI) / math.pi) - r) * 10                     # mm
        mu = torch.log(w_hat.clamp_min(1e-4))
        ls = p['sig0'] + p['sigC'] * torch.log(C) + p['sigR'] * torch.log(r)  # aleatoric SD varies with climate and size
        sicak = (T > p['cH'][:, None]) if s.opt else torch.zeros_like(T, dtype=torch.bool)   # month above the temperature optimum
        return mu, ls, dict(gT=gT, gM=gM, gD=gD, C=C, G=Gp, w=p['omega'] * gEk, sicak=sicak)

def yap(ad, gizli, katman):
    return TeoriModeli19(vpd=True, opt='_opt' in ad, iklim=not ad.startswith('iklimsiz'), agac=ad.endswith('_re'),
                         cwd='_cwd' in ad, spei='_spei' in ad, gizli=gizli, katman=katman).to(DEV)

def ag_param(m):
    return sum(p.numel() for n, p in m.named_parameters() if n != 'u')

In [ ]:
# --- Training and evaluation ---
LOG_Y_SIFIR = math.log(AYAR['sifir_mm']); LOG_ESIK = math.log(0.01)
def hedef(idx):
    w = R_w[idx]
    return torch.where(w > 0, torch.log(w.clamp_min(1e-6)), torch.full_like(w, LOG_Y_SIFIR))

def nll(mu, ls, y):
    """Gaussian NLL of log ring width; zero rings are left-censored below the 0.01 mm resolution."""
    sifir = y <= LOG_Y_SIFIR + 1e-6
    tam = 0.5 * ((y - mu) / torch.exp(ls)) ** 2 + ls + 0.9189385
    sans = -torch.special.log_ndtr((LOG_ESIK - mu) / torch.exp(ls))
    return torch.where(sifir, sans, tam).mean()

@torch.no_grad()
def tahmin(model, idx, bs=131072):
    model.eval(); MU, LS = [], []
    for i in range(0, len(idx), bs):
        mu, ls, _ = model(idx[i:i+bs]); MU.append(mu); LS.append(ls)
    model.train(); return torch.cat(MU), torch.cat(LS)

def yillik_sinyal(idx, mu, y):
    """Median site correlation between observed and predicted growth anomalies, averaged per site and year
    (sites with >= 8 test years). 'tum': each series centred on its test-period mean, so slow size/age trends and
    differences between blocks count. 'blok_ici': each series centred within each 5-year test block, which removes
    trends and levels and keeps only year-to-year (high-frequency, mainly climatic) variation."""
    nz = (y > LOG_Y_SIFIR + 1e-6).cpu().numpy()
    df = pd.DataFrame(dict(ser=R_ser[idx].cpu().numpy(), s=R_site[idx].cpu().numpy(), yil=R_yil[idx].cpu().numpy(),
                           b=R_blok[idx].cpu().numpy(), o=y.cpu().numpy(), p=mu.float().cpu().numpy()))[nz]
    sonuc = {}
    for ad, grup in [('tum', ['ser']), ('blok_ici', ['ser', 'b'])]:
        a = df[['s', 'yil']].copy()
        a['o'] = df.o - df.groupby(grup).o.transform('mean'); a['p'] = df.p - df.groupby(grup).p.transform('mean')
        sy = a.groupby(['s', 'yil'])[['o', 'p']].mean().reset_index()
        r = sy.groupby('s').apply(lambda g: g.o.corr(g.p) if len(g) >= 8 else np.nan, include_groups=False).dropna()
        sonuc[ad] = (float(r.median()), int(len(r)))
    return sonuc

In [ ]:
# --- Load the final model: 4 year folds x 5 seeds (Defter 20) ---
import glob, gc, re
f_ = os.path.join(KAYNAK, 'P2_secilen_mimari.json')
GIZLI, KATMAN = (json.load(open(f_))['gizli'], json.load(open(f_))['katman']) if os.path.exists(f_) else (128, 2)

def yukle(kl):
    m = yap(AYAR['model'], GIZLI, KATMAN); m.load_state_dict(torch.load(os.path.join(kl, 'en_iyi.pt'), map_location=DEV)); m.eval()
    return m

UYE = []
for kl in sorted(glob.glob(os.path.join(KAYNAK, f'yil*__{AYAR["model"]}__h{GIZLI}x{KATMAN}__s*'))):
    if os.path.exists(os.path.join(kl, 'bitti.json')) and os.path.exists(os.path.join(kl, 'en_iyi.pt')):
        k_, s_ = re.search(r'yil(\d+)__.*__s(\d+)$', kl).groups(); UYE.append(dict(kat=int(k_), tohum=int(s_), kl=kl))
assert UYE, f'No trained models in {KAYNAK}: run Defter 20 first'
MODELLER = [yukle(u['kl']) for u in UYE]
ALT = [i for i, u in enumerate(UYE) if u['tohum'] == min(v['tohum'] for v in UYE)]   # one member per fold (limiting factors, curves)
print(f'Final model {AYAR["model"]} ({KATMAN} x {GIZLI}): {len(MODELLER)} members; subset for Q4-Q5: {len(ALT)}')

@torch.no_grad()
def ongor(m, idx, bs=131072, ek=False):
    """Predictions with the current global climate tensor OZ. ek=True also returns per-ring limiting-factor shares."""
    m.eval(); MU, EK = [], []
    for i in range(0, len(idx), bs):
        mu, ls, e = m(idx[i:i+bs]); MU.append(mu)
        if ek:
            w = e['w']; ws = w.sum(1).clamp_min(1e-9)
            st = torch.stack([e['gT'], e['gM'], e['gD']], -1).argmin(-1)                # 0 temperature, 1 soil moisture, 2 VPD
            pay = lambda msk: (w * msk).sum(1) / ws
            EK.append(torch.stack([pay((st == 0) & ~e['sicak']), pay((st == 0) & e['sicak']), pay(st == 1), pay(st == 2), e['C']], 1))
    return torch.cat(MU), (torch.cat(EK) if ek else None)

# site-level groups used throughout
site_bolge = np.array(soz['site_bolge'])
gen_ad = np.array(soz['genus'] + ['Unknown'])[np.minimum(d['site_genus'], len(soz['genus']))]
soguk = desc[:, 2] <= np.quantile(desc[:, 2], 0.25); kurak = desc[:, 5] <= np.quantile(desc[:, 5], 0.25)
nemli = desc[:, 5] >= np.quantile(desc[:, 5], 0.75)
GRUP = {'All sites': np.ones(S, bool), 'Coldest 25%': soguk, 'Driest 25%': kurak, 'Wettest 25%': nemli,
        **{f'Region: {b}': site_bolge == b for b in np.unique(site_bolge)}}
cs_ = pd.Series(gen_ad).value_counts()
GRUP_CINS = {f'Genus: {g}': gen_ad == g for g in cs_[cs_ >= 30].index if g != 'Unknown'}
yuzde = lambda x: 100 * (np.exp(x) - 1)
SITE_n = R_site.cpu().numpy(); YIL_n = R_yil.cpu().numpy()
Y_all = hedef(torch.arange(NR, device=DEV)).cpu().numpy(); NZ_all = Y_all > LOG_Y_SIFIR + 1e-6

In [ ]:
# --- Q0. How much did climate change? (a) CRU: 1990-2019 vs 1901-1960; (b) ISIMIP3a: observed minus counterfactual climate ---
# Counterfactual climate as in Defter 11: the ISIMIP3a obsclim - counterclim difference (Mengel et al. 2021 detrending) is applied
# to the CRU climate the model was trained on; soil moisture is recomputed with the same VS-Lite bucket.
def gun_uzunlugu(lat):
    gun = np.array([15, 46, 74, 105, 135, 166, 196, 227, 258, 288, 319, 349])
    dek = 23.45 * np.sin(np.deg2rad(360 * (284 + gun) / 365))
    x = -np.tan(np.deg2rad(lat))[:, None] * np.tan(np.deg2rad(dek))[None, :]
    return 24 / np.pi * np.arccos(np.clip(x, -1, 1))
def isi_indeksi(T):
    Tclim = np.nanmean(T.reshape(T.shape[0], -1, 12), axis=1)
    return np.sum(np.where(Tclim > 0, (np.maximum(Tclim, 0) / 5) ** 1.514, 0), axis=1)
def kova(T, P, lat, I, Mmax=0.76, Mmin=0.01, alph=0.093, m_th=4.886, mu_th=5.80, rootd=1000., M0=0.2):
    S_, N = T.shape; gu = gun_uzunlugu(lat)
    ngun = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]); L = (ngun / 30)[None, :] * (gu / 12)
    a = 6.75e-7 * I**3 - 7.71e-5 * I**2 + 1.792e-2 * I + 0.49
    M = np.empty((S_, N), np.float32); m = np.full(S_, M0)
    for k in range(N):
        Tt, Pt, Lt = T[:, k], P[:, k], L[:, k % 12]
        Ep = np.where(Tt <= 0, 0., np.where(Tt < 26.5, 16 * Lt * (10 * np.maximum(Tt, 0) / np.maximum(I, 1e-6)) ** a, -415.85 + 32.25 * Tt - 0.43 * Tt**2))
        Ep = np.maximum(Ep, 0); E = Ep * (m / Mmax)
        G_ = mu_th * alph / (1 + mu_th) * m * rootd; R_ = Pt * (m / Mmax) ** m_th + (alph / (1 + mu_th)) * m * rootd
        m = np.clip(m + (Pt - E - R_ - G_) / rootd, Mmin, Mmax); M[:, k] = m
    return M

KL_IS = os.path.join(ANA, 'IKLIM_ISIMIP'); YIL_SON = AYAR['yil_son']
kod2s = {c: i for i, c in enumerate(soz['site_code'])}
def isimip_oku(sen):
    f = os.path.join(KL_IS, f'isimip_{sen}_aylik.csv.gz')
    assert os.path.exists(f), f'Missing {f} (prepared for Defter 11 with ISIMIP_cikar.R)'
    df = pd.read_csv(f, usecols=['studyCode', 'year', 'month', 'tmp', 'pre', 'vpd']); df = df[df.studyCode.isin(list(kod2s))]
    s_ = df.studyCode.map(kod2s).to_numpy(); y_ = df.year.to_numpy() - YIL0; m_ = df.month.to_numpy() - 1
    out = {}
    for v in ['tmp', 'pre', 'vpd']:
        a = np.full((S, NY, 12), np.nan, np.float32); a[s_, y_, m_] = df[v].to_numpy(np.float32); out[v] = a
    return out
OBS = isimip_oku('obsclim'); CFI = isimip_oku('counterclim')
isimip_var = ~np.isnan(OBS['tmp']).all(axis=(1, 2))
dT = np.nan_to_num(OBS['tmp'] - CFI['tmp'])
oranP = np.clip(np.nan_to_num(np.where(OBS['pre'] > 1.0, CFI['pre'] / np.maximum(OBS['pre'], 1e-6), 1.0), nan=1.0), 0.3, 3.0)
dV = np.nan_to_num(OBS['vpd'] - CFI['vpd']); del OBS, CFI; gc.collect()
LATs = d['site_lat'].astype(float)
T_f = np.nan_to_num(clim[..., 0].astype(np.float64)); P_f = np.nan_to_num(clim[..., 1].astype(np.float64)); V_f = np.nan_to_num(clim[..., 3].astype(np.float64))
T_c = T_f - dT; P_c = P_f * oranP; V_c = np.maximum(V_f - dV, 0)
I_tam = isi_indeksi(T_f.reshape(S, -1))
M_f = kova(T_f.reshape(S, -1), P_f.reshape(S, -1), LATs, I_tam); M_c = kova(T_c.reshape(S, -1), P_c.reshape(S, -1), LATs, I_tam)
print('Check: recomputed factual soil moisture vs Defter 1 (max abs diff):', round(float(np.nanmax(np.abs(M_f - soil.reshape(S, -1)))), 4))
OZ_F = OZ
def degistir(kanal):
    Z = OZ_F.clone()
    for c, a in kanal.items():
        a = t(np.asarray(a, np.float32).reshape(S, -1)); Z[..., c] = torch.where(torch.isnan(a), Z[..., c], a)
    return Z
# counterfactual soil moisture as a delta on the factual (Defter 1) soil moisture, so that years without a climate-change
# signal are identical in both scenarios even if the recomputed bucket differs slightly from Defter 1
M_cd = np.clip(soil.reshape(S, -1) + (M_c - M_f), 0.01, 0.76)
OZ_C = degistir({0: T_c, 1: np.log1p(np.maximum(P_c, 0)), 2: M_cd, 3: V_c})      # all channels counterfactual
OZ_T = degistir({0: T_c}); OZ_W = degistir({2: M_cd}); OZ_D = degistir({3: V_c})  # one channel at a time
SENARYO = {'F': OZ_F, 'C': OZ_C, 'T': OZ_T, 'W': OZ_W, 'D': OZ_D}

# (a) CRU change, 1990-2019 vs 1901-1960; (b) climate-change signal (observed - counterfactual), 1990-2019
a0, a1, b0, b1 = 0, 60, 1990 - YIL0, YIL_SON - YIL0 + 1
Mf3, Mc3 = M_f.reshape(S, NY, 12), M_c.reshape(S, NY, 12)
Q0 = pd.DataFrame(dict(studyCode=soz['site_code'], lat=d['site_lat'], lon=d['site_lon'], region=site_bolge, genus=gen_ad, isimip=isimip_var,
    cru_dT_C=T_f[:, b0:b1].mean((1, 2)) - T_f[:, a0:a1].mean((1, 2)),
    cru_dP_pct=100 * (P_f[:, b0:b1].sum(2).mean(1) / np.maximum(P_f[:, a0:a1].sum(2).mean(1), 1e-6) - 1),
    cru_dVPD_kPa=V_f[:, b0:b1].mean((1, 2)) - V_f[:, a0:a1].mean((1, 2)),
    cru_dM=Mf3[:, b0:b1].mean((1, 2)) - Mf3[:, a0:a1].mean((1, 2)),
    cc_dT_C=dT[:, b0:b1].mean((1, 2)),
    cc_dP_pct=100 * (P_f[:, b0:b1].sum((1, 2)) / np.maximum(P_c[:, b0:b1].sum((1, 2)), 1e-6) - 1),
    cc_dVPD_kPa=dV[:, b0:b1].mean((1, 2)), cc_dM=(Mf3 - Mc3)[:, b0:b1].mean((1, 2))))
Q0.to_csv(os.path.join(CIKTI, 'Q0_iklim_degisimi_alan.csv'), index=False)
kol = [c for c in Q0.columns if c.startswith(('cru_', 'cc_'))]
q0g = pd.DataFrame([dict(group=g, n_sites=int(m_.sum()), **{c: float(Q0.loc[m_, c].median()) for c in kol}) for g, m_ in {**GRUP, **GRUP_CINS}.items() if m_.sum() >= 20])
q0g.to_csv(os.path.join(CIKTI, 'Q0_iklim_degisimi_grup.csv'), index=False)
# by decade: anomaly against 1901-1960 (median over sites) and the climate-change signal
on = []
for dc in range(1900, 2030, 10):
    j0, j1 = max(dc - YIL0, 0), min(dc + 10 - YIL0, NY)
    for g, m_ in GRUP.items():
        on.append(dict(decade=dc, group=g, cru_dT_C=float(np.median(T_f[m_, j0:j1].mean((1, 2)) - T_f[m_, a0:a1].mean((1, 2)))),
                       cru_dM=float(np.median(Mf3[m_, j0:j1].mean((1, 2)) - Mf3[m_, a0:a1].mean((1, 2)))),
                       cru_dVPD_kPa=float(np.median(V_f[m_, j0:j1].mean((1, 2)) - V_f[m_, a0:a1].mean((1, 2)))),
                       cc_dT_C=float(np.median(dT[m_, j0:j1].mean((1, 2)))), cc_dM=float(np.median((Mf3 - Mc3)[m_, j0:j1].mean((1, 2))))))
pd.DataFrame(on).to_csv(os.path.join(CIKTI, 'Q0_iklim_degisimi_onyil.csv'), index=False)
print(q0g.round(3).to_string(index=False))

In [ ]:
# --- Q1. Unseen-site test with the uncertainty of unknown trees: for trees never seen in training the tree effect is unknown,
#     so its spread across trained trees (sigma_u) is added to the predictive variance and its mean is used as offset ---
q1 = []
for k in range(5):
    kl = os.path.join(KAYNAK, f'saha{k}__{AYAR["model"]}__h{GIZLI}x{KATMAN}__s0')
    if not os.path.exists(os.path.join(kl, 'en_iyi.pt')): continue
    m = yukle(kl); egt, _, tst = bolunme_saha(k)
    gor = torch.unique(R_tree[egt]); uu = m.u.detach()[gor]; su_, ub_ = float(uu.std()), float(uu.mean())
    y = hedef(tst); nz = y > LOG_Y_SIFIR + 1e-6
    mu0, ls0 = tahmin(m, tst); sg0 = torch.exp(ls0)
    yeni = torch.ones(NTREE, dtype=torch.bool, device=DEV); yeni[gor] = False
    with torch.no_grad(): m.u.data[yeni] = ub_
    mu1, ls1 = tahmin(m, tst); sg1 = torch.sqrt(torch.exp(2 * ls1) + su_ ** 2)
    with torch.no_grad(): m.u.data[yeni] = 0.
    for ad, mu, sg in [('before', mu0, sg0), ('after', mu1, sg1)]:
        e = (mu - y)[nz]
        q1.append(dict(site_group=k, version=ad, sigma_u=su_, mean_u=ub_, n_test=len(tst), r2_log=float(1 - e.pow(2).mean() / y[nz].var()),
                       bias_log=float(e.mean()), nll=float(nll(mu, torch.log(sg), y)),
                       coverage90=float(((y - mu).abs() < 1.645 * sg)[nz].float().mean()), coverage50=float(((y - mu).abs() < 0.674 * sg)[nz].float().mean())))
    del m; torch.cuda.empty_cache()
if q1:
    q1 = pd.DataFrame(q1); q1.to_csv(os.path.join(CIKTI, 'Q1_saha_kalibrasyon.csv'), index=False)
    print(q1.groupby('version')[['r2_log', 'nll', 'coverage90', 'coverage50']].mean().round(3).to_string())

In [ ]:
# --- Q2. Where does the decadal bias come from? Out-of-fold residuals (observed - predicted, log) of Defter 20 ---
oof = np.load(os.path.join(KAYNAK, f'OOF_{AYAR["model"]}.npz'))['mu'].astype(np.float32)
res = Y_all - oof; ok2 = NZ_all & np.isfinite(oof)
ser_n = R_ser.cpu().numpy()
ham = pd.DataFrame(dict(s=d['ring_series'], y=d['ring_year']))
ilk = ham.groupby('s').y.min().reindex(range(NSER)).to_numpy(); son = ham.groupby('s').y.max().reindex(range(NSER)).to_numpy(); del ham
def kat(kod, etiket):
    """Memory-light categorical column from integer codes (19 million rings)"""
    return pd.Categorical.from_codes(np.asarray(kod, np.int16), categories=list(etiket))
def site_kat(lbl):
    c = pd.Categorical(lbl); return kat(c.codes[SITE_n], c.categories)
yas = YIL_n - ilk[ser_n]; kesik = ilk[ser_n] <= YIL0                   # series that start at 1901 have unknown age
yk = np.searchsorted([25, 50, 100, 200], yas, side='left'); yk = np.where(kesik, 5, yk)
yas_c = kat(yk, ['0-25', '26-50', '51-100', '101-200', '>200', 'unknown (starts 1901)']); del yk
boyut_c = kat(np.searchsorted([5, 10, 20, 40], R_rp.cpu().numpy(), side='left'), ['<5 cm', '5-10', '10-20', '20-40', '>40'])
orn_c = kat(np.searchsorted([1969, 1989, 2004], son, side='left')[ser_n], ['<1970', '1970-89', '1990-2004', '>=2005'])
cins_site = np.where(np.isin(gen_ad, list(cs_.index[:12])), gen_ad, 'Other')
ceyrek_site = lambda x: np.array(['Q1', 'Q2', 'Q3', 'Q4'])[np.clip(np.searchsorted(np.quantile(x, [0.25, 0.5, 0.75]), x, side='right'), 0, 3)]
DF2 = pd.DataFrame(dict(res=res, decade=np.minimum(YIL_n // 10 * 10, 2020).astype(np.int16), year=YIL_n.astype(np.int16),
                        region=site_kat(site_bolge), genus=site_kat(cins_site),
                        aridity_quartile=site_kat(ceyrek_site(desc[:, 5])), temperature_quartile=site_kat(ceyrek_site(desc[:, 2])),
                        series_age=yas_c, radius=boyut_c, sampling_end=orn_c))[ok2]
q2, q2e = [], []
for v in ['region', 'genus', 'aridity_quartile', 'temperature_quartile', 'series_age', 'radius', 'sampling_end']:
    g = DF2.groupby([v, 'decade'], observed=True).res.agg(['size', 'mean']).reset_index().rename(columns={v: 'level', 'size': 'n', 'mean': 'mean_residual'})
    g.insert(0, 'variable', v); q2.append(g)
    for lev, x in DF2.groupby(v, observed=True):
        if len(x) < 1000: continue
        b = np.polyfit((x.year.to_numpy() - 1960) / 100, x.res.to_numpy(), 1)[0]
        q2e.append(dict(variable=v, level=lev, n=len(x), slope_log_per_century=float(b), mean_residual=float(x.res.mean())))
b_all = np.polyfit((DF2.year.to_numpy() - 1960) / 100, DF2.res.to_numpy(), 1)[0]
q2e.append(dict(variable='all', level='all', n=len(DF2), slope_log_per_century=float(b_all), mean_residual=float(DF2.res.mean())))
f_ik = os.path.join(KAYNAK, 'OOF_iklimsiz_re.npz')
if os.path.exists(f_ik):
    r_ik = (Y_all - np.load(f_ik)['mu'].astype(np.float32))[ok2]
    q2e.append(dict(variable='all (climate-free model)', level='all', n=len(r_ik),
                    slope_log_per_century=float(np.polyfit((DF2.year.to_numpy() - 1960) / 100, r_ik, 1)[0]), mean_residual=float(r_ik.mean())))
pd.concat(q2).to_csv(os.path.join(CIKTI, 'Q2_sapma_grup.csv'), index=False)
q2e = pd.DataFrame(q2e); q2e.to_csv(os.path.join(CIKTI, 'Q2_sapma_egim.csv'), index=False)
DF2[DF2.series_age != 'unknown (starts 1901)'].groupby(['series_age', 'decade'], observed=True).res.agg(['size', 'mean']).reset_index() \
    .to_csv(os.path.join(CIKTI, 'Q2_sapma_yas_onyil.csv'), index=False)
print(q2e.round(4).to_string(index=False))
del DF2, res, yas, yas_c, boyut_c, orn_c; gc.collect()

In [ ]:
# --- Q3. Attribution: every ring 1902-2019 predicted with factual and counterfactual climate, by every member;
#     channels: temperature only (T), soil moisture only (W), VPD only (D). Effect = log w(factual) - log w(counterfactual). ---
HALKA = ((R_yil >= YIL0 + 1) & (R_yil <= YIL_SON)).nonzero().squeeze(1)
if AYAR['hizli_deneme']: HALKA = HALKA[torch.randperm(len(HALKA), device=DEV)[:300_000]].sort().values
NYa = YIL_SON - YIL0 + 1
ANAHTAR = R_site[HALKA] * NYa + (R_yil[HALKA] - YIL0)
SAYI = torch.bincount(ANAHTAR, minlength=S * NYa).float()
def alan_yil(v): return (torch.bincount(ANAHTAR, weights=v.float(), minlength=S * NYa) / SAYI.clamp_min(1)).reshape(S, NYa).cpu().numpy().astype(np.float32)
SAY = SAYI.reshape(S, NYa).cpu().numpy(); VAR = SAY > 0
KLA = os.path.join(CIKTI, 'atif' + ('_deneme' if AYAR['hizli_deneme'] else '')); os.makedirs(KLA, exist_ok=True)
n_kanal = len(MODELLER) if AYAR['kanal_uyeler'] is None else AYAR['kanal_uyeler']
print(f'{len(HALKA):,} rings, {int(VAR.any(1).sum())} sites; channel runs with {n_kanal} members')
t0 = time.time()
for i, m in enumerate(MODELLER):
    yol = os.path.join(KLA, f'uye{i:02d}.npz')
    if os.path.exists(yol): continue
    sen = ['F', 'C'] + (['T', 'W', 'D'] if i < n_kanal else [])
    mu = {}
    for k_ in sen:
        OZ = SENARYO[k_]; mu[k_] = ongor(m, HALKA)[0]
    OZ = OZ_F
    np.savez_compressed(yol, **{f'fark_{k_}': alan_yil(mu['F'] - mu[k_]) for k_ in sen if k_ != 'F'})
    del mu; torch.cuda.empty_cache()
    print(f'member {i + 1}/{len(MODELLER)} done ({(time.time() - t0) / 60:.1f} min)')
OZ = OZ_F
FK = {k_: [] for k_ in 'CTWD'}
for i in range(len(MODELLER)):
    z = np.load(os.path.join(KLA, f'uye{i:02d}.npz'))
    for k_ in 'CTWD':
        if f'fark_{k_}' in z: FK[k_].append(z[f'fark_{k_}'])
FK = {k_: np.stack(v) for k_, v in FK.items() if v}                    # [members, S, NYa]
KANAL_AD = {'C': 'total', 'T': 'temperature', 'W': 'soil moisture', 'D': 'VPD'}

def donem_ort(A, y0, y1, nmin=3):
    """Site mean over years with data in [y0, y1]; A: [..., S, NYa]"""
    a, b = max(y0 - YIL0, 0), min(y1 - YIL0 + 1, NYa); v = VAR[:, a:b]; n = v.sum(1)
    return np.where(n >= nmin, (A[..., a:b] * v).sum(-1) / np.maximum(n, 1), np.nan)

rngB = np.random.default_rng(2031)
def ozet(E_uye, msk, boot=True):
    """E_uye: [members, S] site effects (log). Group median of the ensemble mean; member 5-95%; site bootstrap 95% CI"""
    e = E_uye.mean(0); ok = msk & np.isfinite(e)
    if ok.sum() < 10: return None
    x = e[ok]; um = np.nanmedian(E_uye[:, ok], axis=1)
    o = dict(n_sites=int(ok.sum()), median_pct=float(yuzde(np.median(x))), mean_pct=float(yuzde(x.mean())),
             members_p05=float(yuzde(np.quantile(um, 0.05))), members_p95=float(yuzde(np.quantile(um, 0.95))),
             share_negative=float((x < 0).mean()),
             share_confident_negative=float((np.quantile(E_uye[:, ok], 0.95, axis=0) < 0).mean()) if len(E_uye) > 1 else np.nan,
             share_confident_positive=float((np.quantile(E_uye[:, ok], 0.05, axis=0) > 0).mean()) if len(E_uye) > 1 else np.nan)
    if boot:
        bs = np.array([np.median(rngB.choice(x, len(x))) for _ in range(500)])
        o.update(boot_lo95=float(yuzde(np.quantile(bs, 0.025))), boot_hi95=float(yuzde(np.quantile(bs, 0.975))))
    return o

# fixed-composition set: sites with >= 50 years of rings in 1951-2010 (so changes are not changes in the site sample)
SABIT = VAR[:, 1951 - YIL0:2011 - YIL0].sum(1) >= 50
q3 = []
for dc in range(1900, 2020, 10):
    for k_, A in FK.items():
        E = donem_ort(A, max(dc, YIL0 + 1), dc + 9)
        for g, msk in GRUP.items():
            for kume, km in [('all_sites', np.ones(S, bool)), ('fixed_set', SABIT)]:
                o = ozet(E, msk & km, boot=(k_ == 'C' and kume == 'all_sites'))
                if o: q3.append(dict(decade=dc, channel=KANAL_AD[k_], group=g, site_set=kume, **o))
q3 = pd.DataFrame(q3); q3.to_csv(os.path.join(CIKTI, 'Q3_atif_onyil.csv'), index=False)

DONEM = [(1931, 1960), (1961, 1990), (1990, YIL_SON)]
q3a = pd.DataFrame(dict(studyCode=soz['site_code'], lat=d['site_lat'], lon=d['site_lon'], region=site_bolge, genus=gen_ad))
q3g, q3k = [], []
for (y0, y1) in DONEM:
    Ed = {k_: donem_ort(A, y0, y1, nmin=10) for k_, A in FK.items()}
    for k_, E in Ed.items():
        q3a[f'{KANAL_AD[k_]} {y0}-{y1} (%)'] = yuzde(E.mean(0))
        if len(E) > 1:
            q3a[f'{KANAL_AD[k_]} {y0}-{y1} members p05 (%)'] = yuzde(np.quantile(E, 0.05, axis=0))
            q3a[f'{KANAL_AD[k_]} {y0}-{y1} members p95 (%)'] = yuzde(np.quantile(E, 0.95, axis=0))
    for g, msk in {**GRUP, **GRUP_CINS}.items():
        for k_, E in Ed.items():
            o = ozet(E, msk, boot=(k_ == 'C'))
            if o: q3g.append(dict(period=f'{y0}-{y1}', channel=KANAL_AD[k_], group=g, **o))
        if all(k_ in Ed for k_ in 'CTWD'):
            ok = msk & np.isfinite(Ed['C'].mean(0)) & np.isfinite(Ed['T'].mean(0))
            if ok.sum() >= 10:
                med = {KANAL_AD[k_]: float(yuzde(np.median(Ed[k_][:len(Ed['T'])].mean(0)[ok]))) for k_ in 'CTWD'}
                q3k.append(dict(period=f'{y0}-{y1}', group=g, n_sites=int(ok.sum()), **med,
                                sum_of_channels=med['temperature'] + med['soil moisture'] + med['VPD'],
                                interaction=med['total'] - (med['temperature'] + med['soil moisture'] + med['VPD'])))
q3a = q3a.merge(Q0, on=['studyCode', 'lat', 'lon', 'region', 'genus'], how='left')
q3a.to_csv(os.path.join(CIKTI, 'Q3_atif_alan.csv'), index=False)
q3g = pd.DataFrame(q3g); q3g.to_csv(os.path.join(CIKTI, 'Q3_atif_grup.csv'), index=False)
if q3k: pd.DataFrame(q3k).to_csv(os.path.join(CIKTI, 'Q3_atif_kanal.csv'), index=False)
print(q3[(q3.channel == 'total') & (q3.group == 'All sites') & (q3.site_set == 'all_sites')]
      [['decade', 'n_sites', 'median_pct', 'members_p05', 'members_p95', 'boot_lo95', 'boot_hi95']].round(2).to_string(index=False))
print(q3g[(q3g.period == f'1990-{YIL_SON}') & (q3g.channel == 'total')][['group', 'n_sites', 'median_pct', 'members_p05', 'members_p95', 'share_negative']]
      .round(2).to_string(index=False))
if q3k: print(pd.DataFrame(q3k).query(f"period == '1990-{YIL_SON}'").round(2).to_string(index=False))

In [ ]:
# --- Q4. Limiting factors and regime shifts: share of the growing-season weight (omega x g_E) limited by cold (temperature below the
#     optimum), heat (temperature above the optimum), soil moisture or VPD; site x period, factual climate, one member per fold ---
HALKA4 = (R_yil >= YIL0 + 1).nonzero().squeeze(1)
if AYAR['hizli_deneme']: HALKA4 = HALKA4[torch.randperm(len(HALKA4), device=DEV)[:300_000]].sort().values
NY4 = NY
K4 = R_site[HALKA4] * NY4 + (R_yil[HALKA4] - YIL0); N4 = torch.bincount(K4, minlength=S * NY4).float()
PAY = 0
for i in ALT:
    _, e = ongor(MODELLER[i], HALKA4, ek=True); PAY = PAY + e / len(ALT); del e
PAY_SY = np.stack([(torch.bincount(K4, weights=PAY[:, j], minlength=S * NY4) / N4.clamp_min(1)).reshape(S, NY4).cpu().numpy() for j in range(5)], -1)
V4 = (N4.reshape(S, NY4) > 0).cpu().numpy(); del PAY
FAK = ['cold', 'heat', 'soil moisture', 'VPD']
DON4 = [(1902, 1930), (1931, 1960), (1961, 1990), (1991, 2019)]
q4 = pd.DataFrame(dict(studyCode=soz['site_code'], lat=d['site_lat'], lon=d['site_lon'], region=site_bolge, genus=gen_ad))
for (y0, y1) in DON4:
    a, b = y0 - YIL0, y1 - YIL0 + 1; v = V4[:, a:b]; n = v.sum(1)
    for j, f in enumerate(FAK + ['C']):
        q4[f'{f} {y0}-{y1}'] = np.where(n >= 5, (PAY_SY[:, a:b, j] * v).sum(1) / np.maximum(n, 1), np.nan)
    q4[f'dominant {y0}-{y1}'] = np.where(n >= 5, np.array(FAK)[np.nan_to_num(q4[[f'{f} {y0}-{y1}' for f in FAK]].to_numpy(), nan=-1).argmax(1)], None)
q4.to_csv(os.path.join(CIKTI, 'Q4_sinirlayici_alan.csv'), index=False)
q4g = []
for (y0, y1) in DON4:
    for g, msk in {**GRUP, **GRUP_CINS}.items():
        x = q4[msk]; ok = x[f'cold {y0}-{y1}'].notna()
        if ok.sum() < 10: continue
        q4g.append(dict(period=f'{y0}-{y1}', group=g, n_sites=int(ok.sum()), **{f: float(x.loc[ok, f'{f} {y0}-{y1}'].mean()) for f in FAK + ['C']},
                        **{f'dominant share: {f}': float((x.loc[ok, f'dominant {y0}-{y1}'] == f).mean()) for f in FAK}))
q4g = pd.DataFrame(q4g); q4g.to_csv(os.path.join(CIKTI, 'Q4_sinirlayici_grup.csv'), index=False)
gec = q4[q4['dominant 1931-1960'].notna() & q4['dominant 1991-2019'].notna()]
pd.crosstab(gec['dominant 1931-1960'], gec['dominant 1991-2019']).to_csv(os.path.join(CIKTI, 'Q4_rejim_gecisi.csv'))
print(q4g[q4g.group == 'All sites'].round(3).to_string(index=False))
print('Regime transition (dominant factor 1931-1960 -> 1991-2019, number of sites):')
print(pd.crosstab(gec['dominant 1931-1960'], gec['dominant 1991-2019']).to_string())

In [ ]:
# --- Q5. Growth-curve shift: a reference tree (start at the site's initial radius, tree effect zero) grown for 100 years at every site
#     under four climates: factual 1920-2019, counterfactual 1920-2019, 1931-1960 climate repeated, 1990-2019 climate repeated ---
N_EGRI = AYAR['egri_yil']
SEN5 = {'factual 1920-2019': (OZ_F, np.arange(1920, 1920 + N_EGRI)),
        'counterfactual 1920-2019': (OZ_C, np.arange(1920, 1920 + N_EGRI)),
        'climate 1931-1960 repeated': (OZ_F, 1931 + np.arange(N_EGRI) % 30),
        'climate 1990-2019 repeated': (OZ_F, 1990 + np.arange(N_EGRI) % 30)}
s_all = torch.arange(S, device=DEV)
G_s = dict(g=site_genus, sp=site_species, desc=DESC, env=ENV, b=site_biyom, e=site_eko, k=site_kg, tree=torch.zeros(S, dtype=torch.long, device=DEV))
@torch.no_grad()
def buyut(m, Z, yillar):
    m.eval(); agac0 = m.agac; m.agac = False                             # reference tree: tree effect switched off
    rp = torch.zeros(S, device=DEV); r0 = m.parametreler(dict(G_s, X=None))['r0']; cap = torch.empty(S, len(yillar), device=DEV)
    for j, y in enumerate(yillar):
        ay = (int(y) - 1 - YIL0) * 12 + torch.arange(24, device=DEV)
        mu, _, _ = m(None, G=dict(G_s, X=Z[:, ay], rp=rp))
        rp = rp + torch.exp(mu) / 10; cap[:, j] = 2 * (r0 + rp)               # stem diameter inside bark, cm
    m.agac = agac0
    return cap.cpu().numpy()
EGRI = {ad: np.mean([buyut(MODELLER[i], Z, yy) for i in ALT], axis=0) for ad, (Z, yy) in SEN5.items()}
def esik_yili(c, e):
    ok = c >= e; return np.where(ok.any(1), ok.argmax(1) + 1, np.nan)
q5 = pd.DataFrame(dict(studyCode=soz['site_code'], lat=d['site_lat'], lon=d['site_lon'], region=site_bolge, genus=gen_ad, has_rings=VAR.any(1)))
for ad, c in EGRI.items():
    q5[f'{ad}: diameter at 50 yr (cm)'] = c[:, 49]; q5[f'{ad}: diameter at {N_EGRI} yr (cm)'] = c[:, -1]
    for e in AYAR['egri_cap_cm']: q5[f'{ad}: years to {e} cm'] = esik_yili(c, e)
q5.to_csv(os.path.join(CIKTI, 'Q5_buyume_egrisi_alan.csv'), index=False)
q5g, q5c = [], []
for g, msk in {**GRUP, **GRUP_CINS}.items():
    msk = msk & q5.has_rings.to_numpy()
    if msk.sum() < 10: continue
    for ad, c in EGRI.items():
        row = dict(group=g, scenario=ad, n_sites=int(msk.sum()), median_diameter_50yr=float(np.median(c[msk, 49])),
                   median_diameter_end=float(np.median(c[msk, -1])))
        for e in AYAR['egri_cap_cm']:
            yy = esik_yili(c[msk], e); row[f'median_years_to_{e}cm'] = float(np.nanmedian(yy)) if np.isfinite(yy).any() else np.nan
            row[f'share_reaching_{e}cm'] = float(np.isfinite(yy).mean())
        q5g.append(row)
        for a_ in range(0, N_EGRI, 5): q5c.append(dict(group=g, scenario=ad, age=a_ + 1, median_diameter=float(np.median(c[msk, a_]))))
    for (a_, b_) in [('factual 1920-2019', 'counterfactual 1920-2019'), ('climate 1990-2019 repeated', 'climate 1931-1960 repeated')]:
        dd = EGRI[a_][msk, -1] / EGRI[b_][msk, -1] - 1
        q5g.append(dict(group=g, scenario=f'{a_} vs {b_}', n_sites=int(msk.sum()), median_diameter_end=float(100 * np.median(dd)),
                        share_negative=float((dd < 0).mean())))
q5g = pd.DataFrame(q5g); q5g.to_csv(os.path.join(CIKTI, 'Q5_buyume_egrisi_grup.csv'), index=False)
pd.DataFrame(q5c).to_csv(os.path.join(CIKTI, 'Q5_egri_ortalama.csv'), index=False)
print(q5g[q5g.group.isin(['All sites', 'Driest 25%', 'Coldest 25%'])].round(2).to_string(index=False))
print('==== DEFTER 21 TAMAM ====')